The workflow for a data science project will follow these lines:

1. Get and explore the data
2. Build a model 
3. Train the model
4. Save and predict

## 1. Get and Explore the Data
The first step can take quite some time; data quality is often something that needs to be checked, and correlations between data should often be explored and visualized.

This step can be a full project on its own: you clean the data, make sure you can access it properly, and create visualizations and hypothesis to gain insight into the data that can be shown in a dashboard.

The insight in the data is an essential ingredient for deciding on a model.

## 2. Build a model
Based on domain knowledge and a first exploration of the data, a model can be selected.

Sometimes, the relation between features and outcome is very obvious. You might have features that
correlate very high with the outcome variable, and a domain expert confirms that the correlations make sense.

If this is the case, you can often build a simple model. If you expect to have non-linear and complex interactions between the features,
you could use a model that works with non-linear data like a SVM plus kernel, or a random forest.

If you have enough data (as a rule of thumb, a lower threshold of 1000 observations) you can consider a neural network architecture.
If the expected complexity of the data is low, you can use a relative small network.
If you have lots and lots of data with a high complexity, you should consider to increase the complexity of your model too.

How you can build a model, and what suitable models are for different datatypes and situations, will be the subject of the whole course.

## 3. Train the model
Once you created a model, it hasnt learned anything yet. The model must be trained to learn the right connections, a bit like a baby that has to learn about what works and what doesn't.

In this notebook, I will introduce you to PyTorch. Another high level library is Tensorflow, which is used a lot too.
While the interface is comparable, the Tensorflow syntax is a bit more high-level. While this can be an advantage, 
it also has a downside: at the moment you ever need to dive a bit deeper into the architecture itself, it is much harder to
add something new with TensorFlow, compared to PyTorch.

## 4. Save and predict
Finally, you will want to use the trained model to predict new observations.

# Load the data
We will use the fashion MNIST dataset: 70.000 small (28x28) greyscale images of clothing in 10
classes. You will find this dataset a lot in machine learning tutorials. We load it straight from
the huggingface hub, so there is nothing to download by hand.

In [ ]:
from pathlib import Path
import warnings

from datasets import load_dataset

warnings.simplefilter("ignore", UserWarning)

fashion = load_dataset("zalando-datasets/fashion_mnist")
fashion

We now have a `Dataset`. They implement at minimum an `.__getitem__` and `.__len__` function.

In [ ]:
train = fashion["train"]
valid = fashion["test"]
train

To get the data, we can use the `__getitem__` method by calling an index, just like you would do with a list or array.

In [ ]:
x = train[0]
x

This is equivalent to this (but no one does that, obviously. We implement the dunder method to make life easier, not more complex...):


In [ ]:
x = train.__getitem__(0)

So a row is a **dictionary**, not a tuple: the hub hands over the image as a `PIL` image object and
the label as an `int`.

In [ ]:
type(x["image"]), x["image"].size, x["image"].mode, x["label"]

That is a real difference from a pytorch-native dataset, and you have to deal with it: a model wants
a **tensor**, channel-first, with values in a sensible range. Nobody is going to do that conversion
for you, so let's write it — the same `process` function shape as in `03_dataloader.ipynb`.

Three things happen in one line each:

- `np.asarray` turns the PIL image into a `(28, 28)` array of bytes 0-255;
- `/ 255.0` scales it to floats in `[0, 1]`, which is what the network wants to see;
- `unsqueeze(0)` adds the **channel** axis in front, giving `(1, 28, 28)`.

That last one is the channel-first convention from `01_tensors.ipynb` §1.6. Fashion MNIST is
greyscale, so there is exactly one channel; a colour image would have three.

In [ ]:
import numpy as np
import torch


def process(item) -> tuple[torch.Tensor, int]:
    img = np.asarray(item["image"], dtype=np.float32) / 255.0
    return torch.from_numpy(img).unsqueeze(0), item["label"]


img, label = process(train[0])
img.shape, img.dtype, img.min().item(), img.max().item(), label

On the tensor ladder that is one item: rank 3 here, `(C, H, W)`, because an image has spatial
structure the network will use. Batching is what adds the leading axis.

Ok, we want to batch this into a dataloader. From the documentation:

> The Dataset retrieves our dataset's features and labels one sample at a time. While training a
> model, we typically want to pass samples in "minibatches", reshuffle the data at every epoch to
> reduce model overfitting, and use Python's multiprocessing to speed up data retrieval

Which is exactly what you built by hand in `03_dataloader.ipynb`, and what a `DataLoader` gives you
for free — as long as you hand it a `collate_fn` that knows how to turn a list of rows into a batch.

In [ ]:
from typing import cast

from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset


def collate_fn(items):
    X, y = zip(*[process(item) for item in items])
    return torch.stack(X), torch.tensor(y)


trainloader = DataLoader(
    cast(TorchDataset, train), batch_size=64, shuffle=True, collate_fn=collate_fn
)
testloader = DataLoader(
    cast(TorchDataset, valid), batch_size=64, shuffle=False, collate_fn=collate_fn
)
len(trainloader)

Why is the length of the dataloader different from the dataset? We had 60000 items before...

### How long does a batch take?

Careful with how you measure this. The obvious idiom is wrong:

```python
%timeit X, y = next(iter(trainloader))   # <- measures the wrong thing
```

`iter(loader)` *builds a new iterator* -- for a `DataLoader` that means a new sampler, a new
shuffle, and (with `num_workers > 0`) a whole new pool of worker processes. Calling it inside the
timing loop means you are timing the setup, not the batching.

So build the iterator once and time `next` on it. One catch: a `DataLoader` is finite, it stops
after one epoch, and `%timeit` will happily ask for more batches than that. The fix is the
generator pattern from `02_datagenerators.ipynb` -- wrap it so it starts over instead of stopping.

In [ ]:
def endless(loader):
    """a finite DataLoader -> an endless stream of batches"""
    while True:
        yield from loader


batches = endless(trainloader)
next(batches)  # warm up: the first call pays for the setup

%timeit X, y = next(batches)

### The other option: streaming

A `DataLoader` over a downloaded dataset is not the only way to get batches. `datasets` can also
**stream**: `streaming=True` downloads nothing up front and pulls rows over the network as you
iterate (see `03_dataloader.ipynb` §3.6). It feeds a `DataLoader` in exactly the same way.

Check out which one is faster -- and think about *why* before you run it.

In [ ]:
stream = load_dataset("zalando-datasets/fashion_mnist", split="train", streaming=True)
streamloader = DataLoader(cast(TorchDataset, stream), batch_size=64, collate_fn=collate_fn)

streambatches = endless(streamloader)
next(streambatches)

%timeit X, y = next(streambatches)

Streaming is somewhat slower -- roughly 1.5x on my machine -- and it should be: every batch is
decoded out of a stream instead of read from a memory-mapped file that the OS has already cached.

But notice how *small* the difference is, and then notice that both numbers are in milliseconds
while a training step on this model takes tens of them. Neither choice is what makes your training
loop slow. Streaming buys you *not needing the data locally*; you pay for it in latency, and you
give up `len()` and random access.

Fashion MNIST is 30 MB. Download it. Reach for streaming when the dataset is bigger than your disk,
not because streaming sounds more modern.

In [ ]:
X, y = next(iter(trainloader))
X.shape, y.shape

In [ ]:
type(X), X.dtype, y.dtype

So, what do we see here? Our datashape has four dimensions:

- 64: this is the batch size. Every batch has 64 observations; in this case 64 images
- 1: this is the channel. Colorimages typically have 3 channels. Our images have just one color, and thus 1 channel. So images can have more channels (e.g. infrared etc)
- (28,28) : this is the actual image, with dimensions 28x28

Lets visualize the first example, the first image:

In [ ]:
img = X[1]
img.shape

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(img.squeeze(), cmap="gray")

# Create a model

In [ ]:
import torch

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

In [ ]:
from torch import nn
from loguru import logger

logger.info(f"Using {device} device")


# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.units_1 = 512
        self.units_2 = 256
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, self.units_1),
            nn.ReLU(),
            nn.Linear(self.units_1, self.units_2),
            nn.ReLU(),
            nn.Linear(self.units_2, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to("cpu")
from torchinfo import summary

summary(model, input_size=(1, 28, 28))

Hopefully, you recognize the setup from the `linearmodel` notebook. 

- We will `Flatten` the image. That means we will transform our (64, 1, 28, 28) data into (64, 784) shaped data. What we do here, is flattening the image into a one dimensional vector.
- We have a stack of hidden layers. These are essentially dotproducts. Our vector of 784 (28*28) elements is transformed into 512 elements, and then into 10 elements because we have 10 classes.
- in between the linear transformations you can see the activation functions,here a `ReLu` 
- The `forward` method is what is called during training. This gives you control over the flow of information: it is easy to create some parallel flow of data if you want to do something like that.

# Optimizer

We need an optimizer. We will dive into this in later lessons.

For now, it is enough to know this:

Your model makes a prediction. But how does the model know if it is right, or wrong?
And, more specific: how does the model know which weights it needs to modify in order

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
X, y = next(iter(trainloader))
model.to(device)
next(model.parameters()).is_cuda

In [ ]:
yhat = model(X.to(device))  # make a prediction

In [ ]:
loss_fn(yhat, y.to(device))  # calculate the loss

# Learn the weights

Specify a directory for logs

In [ ]:
log_dir = Path("demo").absolute()
log_dir

Get a metric to know how well we are doing

In [ ]:
from mltrainer import metrics

accuracy = metrics.Accuracy()

Create settings for the trainer that tell the model
- what an epoch is (train_steps)
- how many epochs we want to train (epochs)
- what metrics we want to use, both on the train and validation set (metrics)
- how to report (reporttypes) and where to store those reports (logdir)

In [ ]:
from mltrainer import TrainerSettings, ReportTypes, Trainer

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(testloader),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)
settings

With the settings in place, we can set up the trainingloop:
- which model to train
- using which settings
- with a loss function and optimizer for training
- two datasets: one for training, one for validation
- a scheduler to adjust the learning rate when we no longer learn
- the device to train on for hardware acceleration (CPU, GPU, MPS)

In [ ]:
import torch.optim as optim

trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=testloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=str(device),
)

And, lets train the model!

In [ ]:
trainer.loop()

Check the `model.toml` and `settings.toml` files that have been created in the specified `demo/` dir we specified in the settings as `logdir`

# Save the model

You will have the latest model at trainer.model, or just use the old model (which is the same)

If you have a look at the settings.earlystop_kwargs, you can see that save is by default false. If you change this to true, the trainer would have kept track of the best model so far and saved it in between. Because this can take up additional time and in a learning setting like we are in we typically dont really want to save the model for later use, we dont need it here.

However, in a real life setting you probably want the best model!

In [ ]:
settings.earlystop_kwargs

You can save it manually

In [ ]:
modeldir = Path("demo/saved_model")
modelpath = modeldir / "trained_model"
if not modeldir.exists():
    modeldir.mkdir(parents=True)
    torch.save(model, modelpath)
    logger.info(f"Model saved to {modelpath}")

In the case you would have set the earlystop.save to true like this:

In [ ]:
settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(testloader),
    reporttypes=[ReportTypes.TENSORBOARD],
    earlystop_kwargs={"save": True, "verbose": True, "patience": 10},
)

The trainer would have saved checkpoints of the last best model. You can obtain the location of the checkpoint with `trainer.early_stopping.path`:

In [ ]:
assert trainer.early_stopping is not None
trainer.early_stopping.path

# load the model

In [ ]:
# note that I would expect the loaded model to run on mps, but that doesnt work as expected
if device == "mps":
    device = "cpu"
print(f"using {device}")
loaded_model = torch.load(modelpath, map_location=device, weights_only=False)

In [ ]:
# show that all parameters are on the same device, and use acceleration if available
for param in loaded_model.parameters():
    print(param.device)

Get a batch $X$, $y$ and make a prediction $\hat{y}$

In [ ]:
X, y = next(iter(testloader))

In [ ]:
yhat = loaded_model(X.to(device))
loss_fn(yhat, y.to(device))

Check the accuracy:
- for every example we have 10 numbers
- the location with the highest value is the prediction
- we can get the index with `argmax` over dimension 1
- we compare that index with the original number
- This gives us a count of all the correct predictions
- dividing that through the total length gives us the accuracy percentage. 

In [ ]:
acc = (yhat.argmax(dim=1) == y.to(device)).sum() / len(y.to(device))
acc.item() * 100

Note that this is the accuracy for a single batch! 
Get another batch by running next() in the cell above, and calculate the accuracy again.